# 미상환 CB·BW 잔액 대조

한 종목의 DART 정기보고서 기준 잔액과 조회 시점의 SEIBRO 잔액을 비교합니다.
유형·회차·발행일·만기일이 모두 일치하며 양쪽에서 각각 한 건인 경우에만 차액을 계산합니다.
보고서에 나타난 잔액은 보고서의 단위를 원으로 환산합니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/2_bond/02_outstanding_bond_reconciliation.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `SEIBRO_API_KEY`, `DART_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 첫 셀은 패키지를 설치하고, 두 번째 셀은 모든 시나리오에 공통인 준비 코드입니다(접어 두어도 됩니다). SEIBRO Open API 키는 [SEIBRO OpenPlatform](https://seibro.or.kr/OpenPlatform/)에서 신청합니다.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = ['SEIBRO_API_KEY', 'DART_API_KEY']
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
else:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + ", ".join(f"{n}: {_status(n)}" for n in SECRETS))  # noqa: F821

SEIBRO 준비 완료 (로컬) | SEIBRO_API_KEY: 있음, DART_API_KEY: 있음


In [3]:
import os
import re

import pandas as pd
from IPython.display import display

from seibro_api import get_bonds_from_report, get_stock_linked_bond_status

STOCK_CODE = "317770"  # 엑스페릭스
REPORT_PERIOD = None  # 기본은 추출된 최근 기간. 예시 "2025.12"
assert os.getenv("DART_API_KEY"), "DART_API_KEY를 Colab Secrets 또는 .env에 등록하세요."

DATE_COLS = ["발행일", "만기일"]
KEYS = ["사채유형", "회차", *DATE_COLS]
REPORT_COLS = ["회차", "보고서유형", "보고서기간", "기준일", "단위", "미상환잔액", "rcept_no_new"]
LIVE_COLS = ["채권코드", "채권명", "미상환잔액", "상세출처"]
UNITS = {"원": 1, "천원": 1_000, "만원": 10_000, "백만원": 1_000_000,
         "억원": 100_000_000, "십억원": 1_000_000_000}


def normalized_dates(values):
    text = values.astype("string").str.strip()
    text = text.str.replace(r"\s*(?:[./-]|년|월)\s*", "-", regex=True).str.rstrip("일")
    return pd.to_datetime(text, format="mixed", errors="coerce").dt.strftime("%Y-%m-%d")


def report_unit(value):
    text = re.sub(r"\s+", "", str(value))
    currencies = re.findall(r"(?:십억|백만|천|만|억)?원|USD|EUR|JPY|CNY|달러|유로|엔", text, flags=re.I)
    if len(currencies) == 1:
        return UNITS.get(currencies[0])
    return None


def amount(values):
    return pd.to_numeric(values.astype("string").str.replace(",", "").str.strip(), errors="coerce")

In [4]:
source_reports = get_bonds_from_report(STOCK_CODE, save_csv=False)
dart = pd.concat([
    source_reports[kind].reindex(columns=[*DATE_COLS, *REPORT_COLS]).assign(사채유형=kind.upper())
    for kind in ("cb", "bw")
], ignore_index=True)
period = REPORT_PERIOD or (dart["보고서기간"].dropna().max() if dart["보고서기간"].notna().any() else None)
dart = dart.loc[dart["보고서기간"].eq(period)].copy()
dart = dart.dropna(how="all", subset=["회차", "미상환잔액", *DATE_COLS])
dart["회차"] = dart["회차"].astype("string").str.extract(r"^\s*(?:제\s*)?(\d+)", expand=False)
dart["DART 잔액(원)"] = amount(dart["미상환잔액"]) * pd.to_numeric(dart["단위"].map(report_unit), errors="coerce")
dart["DART 원문"] = dart["rcept_no_new"].map(
    lambda number: f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={number}" if pd.notna(number) else ""
)

live = get_stock_linked_bond_status(STOCK_CODE, verbose=False)
live = live.reindex(columns=["구분", *DATE_COLS, *LIVE_COLS]).rename(columns={"구분": "사채유형"})
live = live.loc[live["사채유형"].isin(["CB", "BW"])].copy()
live["회차"] = live["채권명"].astype("string").str.extract(r"(?<!\d)(\d+)\s*(?:CB|BW)\b", flags=re.I, expand=False)
live["SEIBRO 잔액(원)"] = amount(live["미상환잔액"])
live["SEIBRO 조회일"] = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y-%m-%d")

for frame in (dart, live):
    frame["회차"] = pd.to_numeric(frame["회차"], errors="coerce").astype("Int64")
    for column in DATE_COLS:
        frame[column] = normalized_dates(frame[column])


[1/3] 317770 최근 사업/반기보고서 조회 중...
  기간: 20241125 ~ 20261125


  -> 사업보고서: 사업보고서 (2025.12) (20260318)
  -> 반기보고서: 반기보고서 (2026.06) (20260814)

[2/3] rcept_no_new 확보 중...
  반기보고서: 20260814001639


  사업보고서: 20260318001711



[3/3] XML 원문 다운로드 중...


  OK 반기보고서 (20260814001639) - 1,762,808자


  OK 사업보고서 (20260318001711) - 2,321,474자


... (3 outputs left out)

  반기보고서(2026.06): 3건 (메타: {'BASE_DT': '2026년 06월 30일', 'WONSTOCK': '(단위 : 원, 주)'})


  사업보고서(2025.12): 3건 (메타: {'BASE_DT': '2025년 12월 31일', 'WONSTOCK': '(단위 : 원, 주)'})

  총 6건

[파싱] eb (aclass=EB) ...


  반기보고서(2026.06): 데이터 없음


  사업보고서(2025.12): 데이터 없음
  eb 데이터를 찾을 수 없습니다.


In [5]:
duplicate_keys = pd.concat([frame.loc[frame.duplicated(KEYS, keep=False), KEYS] for frame in (dart, live)])
excluded = pd.MultiIndex.from_frame(duplicate_keys)
dart_matches = dart[KEYS].notna().all(axis=1) & ~pd.MultiIndex.from_frame(dart[KEYS]).isin(excluded)
live_matches = live[KEYS].notna().all(axis=1) & ~pd.MultiIndex.from_frame(live[KEYS]).isin(excluded)
dart_view = [*KEYS, "보고서유형", "보고서기간", "기준일", "단위", "DART 잔액(원)", "DART 원문"]
live_view = [*KEYS, "채권코드", "채권명", "SEIBRO 잔액(원)", "SEIBRO 조회일", "상세출처"]
comparison = dart.loc[dart_matches, dart_view].merge(
    live.loc[live_matches, live_view], on=KEYS, how="outer", indicator="매칭", validate="one_to_one",
)
comparison["차액(원)"] = comparison["SEIBRO 잔액(원)"] - comparison["DART 잔액(원)"]
comparison["매칭"] = comparison["매칭"].map({
    "both": "양쪽 확인", "left_only": "DART 한쪽에서 확인", "right_only": "SEIBRO 한쪽에서 확인",
})

print(f"DART 보고서기간 {period or '확인된 표 없음'} · SEIBRO 조회 {pd.Timestamp.now(tz='Asia/Seoul'):%Y-%m-%d}")
display(comparison)
for label, frame, eligible, columns in (
    ("DART", dart, dart_matches, dart_view), ("SEIBRO", live, live_matches, live_view),
):
    if (~eligible).any():
        print(f"{label} · 회차·날짜 누락 또는 중복으로 대조 보류")
        display(frame.loc[~eligible, columns])

DART 보고서기간 2026.06 · SEIBRO 조회 2026-11-25


,사채유형,회차,발행일,만기일,보고서유형,보고서기간,기준일,단위,DART 잔액(원),DART 원문,채권코드,채권명,SEIBRO 잔액(원),SEIBRO 조회일,상세출처,매칭,차액(원)
0,BW,2,2024-02-28,2027-02-28,반기보고서,2026.06,2026년 06월 30일,"(단위 : 원, 주)",4460000000,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,KR6317772E29,엑스페릭스 2BW(사모/신/풋),4460000000,2026-11-25,웹,양쪽 확인,0
1,BW,3,2024-02-28,2027-02-28,반기보고서,2026.06,2026년 06월 30일,"(단위 : 원, 주)",210000000,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,KR6317773E28,엑스페릭스 3BW(사모/신/풋),210000000,2026-11-25,웹,양쪽 확인,0
2,CB,1,2024-02-28,2027-02-28,NaN,NaN,NaN,NaN,<NA>,NaN,KR6317771E20,엑스페릭스 1CB(사모/전환/풋),0,2026-11-25,웹,SEIBRO 한쪽에서 확인,<NA>
3,CB,4,2024-05-23,2027-05-23,반기보고서,2026.06,2026년 06월 30일,"(단위 : 원, 주)",1750000000,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,KR6317771E53,엑스페릭스 4CB(사모/전환/풋),1750000000,2026-11-25,웹,양쪽 확인,0
4,CB,5,2025-08-06,2028-08-06,반기보고서,2026.06,2026년 06월 30일,"(단위 : 원, 주)",4000000000,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,KR6317771F86,엑스페릭스 5CB(사모/전환/풋),4000000000,2026-11-25,웹,양쪽 확인,0
5,CB,6,2026-05-15,2029-05-15,반기보고서,2026.06,2026년 06월 30일,"(단위 : 원, 주)",15500000000,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,KR6317771G51,엑스페릭스 6CB(사모/전환/풋),15500000000,2026-11-25,웹,양쪽 확인,0
6,CB,7,2026-09-21,2029-09-21,NaN,NaN,NaN,NaN,<NA>,NaN,KR6317771G93,엑스페릭스7CB(사모/전환/풋),11000000000,2026-11-25,웹,SEIBRO 한쪽에서 확인,<NA>


잔액 차이는 기준일 사이에 발생한 전환·행사·상환 또는 데이터 반영 시차에서 생길 수 있습니다.
회차나 날짜가 빠졌거나 중복된 사채, 한쪽에서만 확인되는 사채, 통화·단위가 불명확한 금액은 차액을 비워 둡니다.